In [40]:
import os, json
from itertools import product

RAW_PATH = "../data/benchmark/MT745584.txt"   # genome thô hoặc FASTA
TX_PATH  = "../data/processed/covid_kmer.txt"           # file giao dịch (định dạng SPMF) sẽ được sinh ra
MAP_PATH = "../data/processed/covid_kmer_map.json"      # bảng id <-> k-mer
K      = 6      # độ dài k-mer (item)
WINDOW = 100    # độ dài mỗi đoạn genome = 1 giao dịch
STRIDE = WINDOW - K + 1     # bước trượt (STRIDE < WINDOW: các đoạn chồng lấn)
MINSUP = 0.1

# 1. Đọc genome (bỏ dòng header FASTA ">...", nối các dòng lại)
with open(RAW_PATH) as f:
    genome = "".join(l.strip().upper() for l in f if l.strip() and not l.startswith(">"))
print(f"Độ dài genome: {len(genome):,} nucleotide")

# 2. Bảng ánh xạ k-mer -> số nguyên (id cố định, bắt đầu từ 1)
kmer2id = {"".join(p): i for i, p in enumerate(product("ACGT", repeat=K), 1)}

# 3. Mỗi đoạn trượt -> một giao dịch gồm tập id các k-mer xuất hiện trong đoạn
starts, n_skip = [], 0
with open(TX_PATH, "w") as out:
    for s in range(0, len(genome) - WINDOW + 1, STRIDE):
        w = genome[s:s + WINDOW]
        ids = {kmer2id[w[j:j + K]] for j in range(WINDOW - K + 1) if w[j:j + K] in kmer2id}  # bỏ k-mer chứa N...
        if not ids:
            n_skip += 1
            continue
        out.write(" ".join(map(str, sorted(ids))) + "\n")   # SPMF: item tăng dần
        starts.append(s)

with open(MAP_PATH, "w") as f:
    json.dump({"k": K, "window": WINDOW, "stride": STRIDE, "starts": starts,
               "id2kmer": {str(v): k for k, v in kmer2id.items()}}, f)

print(f"Số giao dịch: {len(starts)} (bỏ {n_skip} đoạn rỗng) | k={K}, window={WINDOW}, stride={STRIDE}")
print(f"Đã ghi {TX_PATH}")

Độ dài genome: 29,860 nucleotide
Số giao dịch: 314 (bỏ 0 đoạn rỗng) | k=6, window=100, stride=95
Đã ghi ../data/processed/covid_kmer.txt


In [41]:
!julia --project=.. ../run_cli.jl --input {TX_PATH} --minsup {MINSUP} --algo optimized --output ../results/covid_out.txt

[1/3] Đang đọc CSDL từ: ../data/processed/covid_kmer.txt
[2/3] Đang chạy thuật toán Apriori (optimized) | Minsup tương đối: 0.1 (Tối thiểu 32 giao dịch)
[3/3] Hoàn thành trong: 2.163s. Tìm thấy 12 frequent itemsets.
Đang ghi kết quả ra: ../results/covid_out.txt


  Activating project at `d:\HCMUS\Data Mining\Frequent Itemset\Frequent Itemset\21`


In [42]:
import os, json

output_path = "../results/covid_out.txt"
info = json.load(open("../data/processed/covid_kmer_map.json"))
id2kmer, starts, W = info["id2kmer"], info["starts"], info["window"]
N = len(starts)
MIN_SIZE = 2   # chỉ xét motif gồm >= 2 k-mer (1 k-mer đơn lẻ không mang nhiều ý nghĩa)

frequent_itemsets = []
if os.path.exists(output_path):
    with open(output_path) as f:
        for line in f:
            if "#SUP:" in line:
                left, sup = line.strip().split("#SUP:")
                frequent_itemsets.append((left.split(), int(sup)))

    by_size = {}
    for items, _ in frequent_itemsets:
        by_size[len(items)] = by_size.get(len(items), 0) + 1
    print(f"Tổng số itemset phổ biến: {len(frequent_itemsets)} | theo kích thước: {dict(sorted(by_size.items()))}")

    cands = sorted((x for x in frequent_itemsets if len(x[0]) >= MIN_SIZE),
                   key=lambda x: (-x[1], -len(x[0])))
    print("-" * 78)
    print(f"{'STT':<4} | {'Motif (tổ hợp k-mer)':<40} | {'Size':<4} | {'Support':<8} | {'% đoạn':<7}")
    print("-" * 78)
    for idx, (items, sup) in enumerate(cands[:10], 1):
        motif = " + ".join(id2kmer[i] for i in items)
        print(f"{idx:<4} | {motif:<40} | {len(items):<4} | {sup:<8} | {100 * sup / N:5.1f}%")

    # Vị trí trên genome của motif đứng đầu (để thảo luận ý nghĩa sinh học)
    if cands:
        top = set(cands[0][0])
        pos = []
        with open("../data/processed/covid_kmer.txt") as f:
            for s, line in zip(starts, f):
                if top <= set(line.split()):
                    pos.append(s)
        print(f"\nMotif top-1 xuất hiện ở {len(pos)} đoạn, vị trí genome từ {min(pos)} đến {max(pos) + W}")
else:
    print(f"Chưa tìm thấy {output_path}. Hãy chạy các cell trước!")

Tổng số itemset phổ biến: 12 | theo kích thước: {1: 12}
------------------------------------------------------------------------------
STT  | Motif (tổ hợp k-mer)                     | Size | Support  | % đoạn 
------------------------------------------------------------------------------
